# Lab 3 Track B — Policy-grounded offer generation

This notebook builds the local FAISS index over the four NorthStar policy files, retrieves policy evidence for one customer, generates an offer through Bedrock, and checks the answer against the retrieved policy. Run it from the repository root with the personal `northstar` AWS profile in `us-east-1`.

In [1]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "northstar-policy-docs").is_dir():
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
import boto3
from rag_offer_system import RAGOfferSystem

REGION = "us-east-1"
identity = boto3.client("sts", region_name=REGION).get_caller_identity()
assert identity["Account"] == "573693339848", identity["Account"]
print(f"AWS account verified: {identity['Account']} ({REGION})")
print(f"Repository root: {REPO_ROOT}")

AWS account verified: 573693339848 (us-east-1)
Repository root: /Users/arsabolsky/Library/Mobile Documents/com~apple~CloudDocs/6. Fall 2026/CS 401R Engineering The Ai Enterprise/cs401r-sabolsky


In [2]:
customer = {
    "customer_id": "DEMO-GOLD-01",
    "loyalty_tier": "Gold",
    "churn_probability": 0.81,
    "days_since_last_purchase": 47,
    "total_lifetime_value": 3120.40,
    "avg_order_value": 186.25,
    "category_diversity_score": 0.5,
    "online_to_store_ratio": 0.72,
    "top_categories": ["Outdoor", "Footwear"],
}
question = (
    "Generate a personalized, policy-grounded retention offer for this at-risk customer."
)
system = RAGOfferSystem(
    corpus_dir=REPO_ROOT / "northstar-policy-docs",
    region=REGION,
    retrieval_strategy="C",
    generation_model_id="amazon.nova-pro-v1:0",
)
print(f"Loaded {len(system.chunks)} policy chunks into one FAISS index.")
print(f"Embedding model: {system.embedding_model_id}")
print(f"Generation model: {system.generation_model_id}")

Loaded 51 policy chunks into one FAISS index.
Embedding model: amazon.titan-embed-text-v2:0
Generation model: amazon.nova-pro-v1:0


In [3]:
request, retrieved = system.retrieve(customer, question)
print(f"Retrieval query: {request.query}")
for chunk in retrieved:
    print(f"\n[{chunk.doc_id} | {chunk.section} | {chunk.chunk_id}]")
    print(chunk.text[:700])

Retrieval query: tier Gold entitlements and shipping thresholds for a customer who shops primarily online. Generate a personalized, policy-grounded retention offer for this at-risk customer.

[POL-LOY-011 | 4. Tier Benefits | POL-LOY-011-s10-c01]
4. Tier Benefits
| Benefit | Bronze | Silver | Gold | Platinum | |---|---|---|---|---| | Points earn rate | 1× | 1.25× | 1.5× | 2× | | Birthday reward | $5 | $10 | $20 | $40 | | Free standard shipping | $75+ | $50+ | Always | Always | | Free return shipping | No | No | Yes | Yes | | Return window | 30 days | 30 days | 30 days | **60 days** | | Early access to sales | No | 12 hours | 24 hours | 48 hours | | Free gear repair | No | No | 2 per year | Unlimited | | Dedicated support line | No | No | No | Yes | | Annual gear credit | No | No | No | $100 | Benefits are tied to the member's tier at the time the benefit is used, except the return window, which is fixed at the tier held **at the time

[POL-SHP-007 | 2. Free Shipping Thresholds | POL-SH

In [4]:
result = system.generate_offer_with_metadata(customer, question)
print("Generated offer:")
print(result["answer"])
print(f"Policy fallback used: {result['guardrail_fallback']}")
print(f"Returned chunk IDs: {result['chunk_ids']}")

Generated offer:
If your Gold tier is confirmed under the current trailing-12-month spend rules, the policy lists free standard shipping on any order. Explore the Outdoor category and visit NorthStar to consider your next purchase.
Policy fallback used: True
Returned chunk IDs: ['POL-LOY-011-s10-c01', 'POL-SHP-007-s03-c01', 'POL-LOY-011-s04-c01', 'POL-LOY-011-s06-c01']


In [5]:
answer_lower = result["answer"].lower()
category_grounded = any(category.lower() in answer_lower for category in customer["top_categories"])
policy_grounded = any(
    phrase in answer_lower and any(phrase in context.lower() for context in result["contexts"])
    for phrase in ("free standard shipping", "free return shipping", "early access")
)
conditional_tier = "if your gold tier is confirmed" in answer_lower
print({
    "category_from_profile": category_grounded,
    "benefit_found_in_returned_policy_context": policy_grounded,
    "tier_claim_is_conditional": conditional_tier,
})
assert category_grounded and policy_grounded and conditional_tier

{'category_from_profile': True, 'benefit_found_in_returned_policy_context': True, 'tier_claim_is_conditional': True}
